### **Setup: Imports und Initialisierung**

In [ ]:
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath('..'))

from quinex_application.mongos_store import MongoStore
from quinex_application.quinex import Quinex
from quinex_application.visualisation_quinex import VisualisationQuinex
from quinex_application.oeo_data import OEOData
from quinex_application.levenshtein import LevenshteinDistance
from quinex_application.techdb import TechDB

In [ ]:
quinex = Quinex()
mongodb = MongoStore(quinex, database_name="quinex_db_query4", collection_name="papers_2023")
visualisation = VisualisationQuinex()
oeo = OEOData("../data/oeo/etd.xlsx", "../data/oeo/oeo-full.owl", "../data/oeo/oeo.xlsx")
levenshtein = LevenshteinDistance()
tb = TechDB()

In [ ]:
analysis = mongodb.get_collection("papers_2023")

### **Load Analysis into DataFrame**

In [ ]:
# TODO: Auffüllen für value und unit anpassen in quantity_normalized
# TODO: Modifier/Qualifier auch mitnehmen in quantity_normalized
# TODO: Wenn quantities leer ist bei single_quantity (keine normalisierung geklappt): Type auf literal setzen
df_path = "../data/quinex/df_quinex.parquet"

df_quinex = visualisation.create_df(analysis)

In [ ]:
visualisation.save_df(df_quinex, df_path)

### **Metadata Results**

In [ ]:
# Alle Paper-IDs, die in analysis vorkommen (mit Analyse)
analysis_ids = {paper["_id"] for paper in analysis if paper.get("analysis_result")}

# Alle Paper-IDs, die tatsächlich im DataFrame gelandet sind
df_ids = set(df_quinex["paper_id"].unique())

# Fehlende Paper mit Analyse, aber ohne quantitative Aussage
missing = analysis_ids - df_ids

print(f"🧾 Total analyzed papers (with analysis_result): {len(analysis_ids)}")
print(f"✅ Papers in DataFrame: {len(df_ids)}")
print(f"❌ Papers with no valid quantitative statements: {len(missing)}")

# Optional: welche das sind
print("Missing paper_ids:", missing)

In [ ]:
# Gesamtanzahl der quantitativen Aussagen
total_statements = len(df_quinex)

# Durchschnittliche Anzahl an Aussagen pro Paper
avg_statements_per_paper = total_statements / len(df_ids)

print(f"📊 Total quantitative statements: {total_statements}")
print(f"➗ Average statements per paper: {avg_statements_per_paper:.2f}")

### **Apply Levenshtein Method with TechDB Hirarchy**

In [ ]:
en_list = tb.save_entities()
prop_list = tb.save_properties()
df_quinex = visualisation.load_df("../data/quinex/df_quinex.parquet")

In [ ]:
import requests
url = "http://127.0.0.1:8080/match_dataset"
json_dict = {
    "levenshtein": "similarity",
    "data": df_quinex[['entity', 'property']].head().to_dict(orient="records")
}
response = requests.post(url, json=json_dict)

In [17]:
import pandas as pd
df_response = pd.DataFrame(response.json())
df_mapped = df_quinex.merge(df_response, on=['entity', 'property'], how='left')
display(df_mapped.head())

,paper_id,entity,property,quantity_text,quantity_normalized,quote,temporal_scope_text,spatial_scope_text,entity_mapping,entity_match_score,property_mapping,property_match_score
0,10.1016/j.rser.2025.115648,,carbon footprint,= zero,"{'quantities': [{'modifier': '=', 'unit': [{'exponent': None, 'uri': 'http://qudt.org/vocab/unit/UNITLESS', 'year': None}], 'value': 0.0}], 'type': 'single_quantity'}",ethylene) cannot easily be substituted with zero-carbon alternatives and remain interconnected in a low-carbon future.,,,None,NaN,carbon storage,0.562500
1,10.1016/j.rser.2025.115648,biobased hydrocarbons,costs,< 1.1 USD/kg,"{'quantities': [{'modifier': '<', 'unit': [{'exponent': 1.0, 'uri': 'http://qudt.org/vocab/currency/USD', 'year': None} {'exponent': -1.0, 'uri': 'http://qudt.org/vocab/unit/KiloGM', 'year': None}], 'value': 1.1}], 'type': 'single_quantity'}","This paper estimates the future costs of low-carbon ethylene, propylene, and jet fuel via those routes with feedstocks of either biomass or electricity with captured CO2. It finds while biobased hydrocarbons could fall below 1.1 USD/kg, electricity-based hydrocarbons using atmospheric CO2, even with the optimistic views, result in 4 USD/kg for ethylene, 2.",,,hydrocarbon,0.52381,cost,0.800000
2,10.1016/j.rser.2025.115648,ethylene,future costs,= 4 USD/kg,"{'quantities': [{'modifier': '=', 'unit': [{'exponent': 1.0, 'uri': 'http://qudt.org/vocab/currency/USD', 'year': None} {'exponent': -1.0, 'uri': 'http://qudt.org/vocab/unit/KiloGM', 'year': None}], 'value': 4.0}], 'type': 'single_quantity'}","This paper estimates the future costs of low-carbon ethylene, propylene, and jet fuel via those routes with feedstocks of either biomass or electricity with captured CO2. It finds while biobased hydrocarbons could fall below 1.1 USD/kg, electricity-based hydrocarbons using atmospheric CO2, even with the optimistic views, result in 4 USD/kg for ethylene, 2.",,,methane,0.50000,fuel cost,0.583333
3,10.1016/j.rser.2025.115648,propylene,future costs,= 2.3 USD/kg,"{'quantities': [{'modifier': '=', 'unit': [{'exponent': 1.0, 'uri': 'http://qudt.org/vocab/currency/USD', 'year': None} {'exponent': -1.0, 'uri': 'http://qudt.org/vocab/unit/KiloGM', 'year': None}], 'value': 2.3}], 'type': 'single_quantity'}","This paper estimates the future costs of low-carbon ethylene, propylene, and jet fuel via those routes with feedstocks of either biomass or electricity with captured CO2. It finds while biobased hydrocarbons could fall below 1.1 USD/kg, electricity-based hydrocarbons using atmospheric CO2, even with the optimistic views, result in 4 USD/kg for ethylene, 2.3 USD/kg for propylene and 2.",,,propulsion,0.50000,fuel cost,0.583333
4,10.1016/j.rser.2025.115648,jet fuel,future costs,= 2.9 USD/kg,"{'quantities': [{'modifier': '=', 'unit': [{'exponent': 1.0, 'uri': 'http://qudt.org/vocab/currency/USD', 'year': None} {'exponent': -1.0, 'uri': 'http://qudt.org/vocab/unit/KiloGM', 'year': None}], 'value': 2.9}], 'type': 'single_quantity'}","This paper estimates the future costs of low-carbon ethylene, propylene, and jet fuel via those routes with feedstocks of either biomass or electricity with captured CO2. It finds while biobased hydrocarbons could fall below 1.1 USD/kg, electricity-based hydrocarbons using atmospheric CO2, even with the optimistic views, result in 4 USD/kg for ethylene, 2.3 USD/kg for propylene and 2.9 USD/kg for jet fuel.",,,jet fuel,1.00000,fuel cost,0.583333


In [ ]:
visualisation.save_df(df_mapped, "../data/quinex/df_quinex_levenshtein_extended.parquet")

In [19]:
df_levenshtein = df_mapped[(df_mapped['entity_match_score'] >= 0.75) & (df_mapped['property_match_score'] >=0.75)]
display(df_levenshtein.head(10))
print(len(df_levenshtein))

,paper_id,entity,property,quantity_text,quantity_normalized,quote,temporal_scope_text,spatial_scope_text,entity_mapping,entity_match_score,property_mapping,property_match_score


0


In [ ]:
visualisation.save_df(df_levenshtein, "../data/quinex/df_quinex_prop_ent_mapped_extended.parquet")

In [ ]:
print(df_quinex["modifier"].value_counts())

In [ ]:
print(df_levenshtein["entity_mapping"].nunique())
print(df_levenshtein["property_mapping"].nunique())

### **Visualise Results**

In [ ]:
df_quinex_mapped = pd.read_parquet("../data/quinex/df_quinex_prop_ent_mapped.parquet")
print(len(df_quinex_mapped))

In [ ]:
fig = visualisation.plot_top_values(df_quinex, column="entity", top_n=15, ignore_empty=True, figsize=(6,6), title="Top 15 entities")
fig = visualisation.plot_top_values(df_quinex_mapped, column="entity_mapping", top_n=15, ignore_empty=True, figsize=(6,6), title="Top 15 mapped entities")

In [ ]:
fig = visualisation.plot_top_properties_of_entity(df_quinex, "electrolyzer", top_n=10, ignore_empty=True, figsize=(6,6))
fig = visualisation.plot_top_quantity_of_property(df_quinex, "electrolyzer", "rated power", top_n=8, ignore_empty=True, figsize=(6,6))

In [ ]:
df_path = "../data/quinex/df_quinex_levenshtein.parquet"
fig = visualisation.plot_levenshtein_score_limits(df_path, score_limits=[0.7, 0.75, 0.8, 0.85, 0.9])
fig = visualisation.plot_unique_entities_over_score_limits(df_path, score_limits=[0.7, 0.75, 0.8, 0.85, 0.9])